# YouGov approval: preprocessing

Combine the two YouGov downloads for US registered voters and their subgroups into one dataset for the four study windows. 

## 1. Data loading

Inspect the sheets, table sizes and layout

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "yougov"
OUT = ROOT / "data" / "cleaned" / "yougov_approval_clean.csv"
FILES = ["donald-trump-approval-2017-2021.xlsx", "donald-trump-approval.xlsx"]

In [2]:
workbooks = {name: pd.read_excel(RAW / name, sheet_name=None, header=None, engine="openpyxl")
             for name in FILES}
for name, sheets in workbooks.items():
    print(f"{name}: {len(sheets)} sheets")
    
    display(pd.DataFrame([{"sheet": sheet, "rows": table.shape[0], "columns": table.shape[1]} for sheet, table in sheets.items()]))

donald-trump-approval-2017-2021.xlsx: 18 sheets


,sheet,rows,columns
0,US Registered Voters,6,208
1,Under 30,6,208
2,30-44,6,208
3,45-64,6,208
4,65+,6,208
5,Male,6,208
6,Female,6,208
7,White,6,208
8,Black,6,208
9,Hispanic,6,208


donald-trump-approval.xlsx: 18 sheets


,sheet,rows,columns
0,US Registered Voters,6,96
1,Under 30,6,96
2,30-44,6,96
3,45-64,6,96
4,65+,6,96
5,Male,6,96
6,Female,6,96
7,White,6,96
8,Black,6,96
9,Hispanic,6,96


In [3]:
for name, sheets in workbooks.items():
    print(f"Original data: {name} - US Registered Voters")
    display(sheets["US Registered Voters"].iloc[:, :6])

Original data: donald-trump-approval-2017-2021.xlsx - US Registered Voters


,0,1,2,3,4,5
0,Do you approve or disapprove of the way Donald...,2017-02-07,2017-02-14,2017-02-22,2017-03-01,2017-03-07
1,Approve,0.45,0.46,0.48,0.45,0.44
2,Disapprove,0.49,0.49,0.47,0.51,0.51
3,Not sure,0.06,0.05,0.05,0.04,0.06
4,Unweighted base,1315,1283,1198,1251,1359
5,Base,1086,1100,1031,1051,1168


Original data: donald-trump-approval.xlsx - US Registered Voters


,0,1,2,3,4,5
0,Do you approve or disapprove of the way Donald...,2025-01-28,2025-02-04,2025-02-11,2025-02-18,2025-02-25
1,Approve,0.5,0.48,0.47,0.5,0.48
2,Disapprove,0.46,0.47,0.49,0.47,0.47
3,Not sure,0.04,0.05,0.04,0.03,0.05
4,Unweighted base,1376,1423,1430,1451,1444
5,Base,1148,1167,1172,1174,1173


## 2. Data reshaping

Each row now represents one source date and one group, with approval measures in separate columns. The overall series and overlapping subgroups remain separate.

In [4]:
GROUPS = {
    "US Registered Voters": "overall",
    "Under 30": "age", "30-44": "age", "45-64": "age", "65+": "age",
    "Male": "gender", "Female": "gender",
    "White": "race_ethnicity", "Black": "race_ethnicity", "Hispanic": "race_ethnicity", "Other": "race_ethnicity",
    "HS or less": "education", "Some college": "education", "College grad": "education", "Postgrad": "education",
    "Democrat": "party", "Independent": "party", "Republican": "party",
}
MEASURES = {"Approve": "approve", "Disapprove": "disapprove", "Not sure": "not_sure",
            "Unweighted base": "unweighted_base", "Base": "base"}

In [5]:
frames = []
for name, sheets in workbooks.items():
    for group, raw in sheets.items():
        frame = raw.iloc[1:, 1:].T.copy()
        frame.columns = raw.iloc[1:, 0].str.strip().map(MEASURES)
        frame = frame.reset_index(drop=True)
        frame["date"] = pd.to_datetime(raw.iloc[0, 1:].to_numpy(), format="%Y-%m-%d")
        frame["group_type"] = GROUPS[group]
        frame["group"] = group
        frame["source_file"] = name
        frames.append(frame)

clean = pd.concat(frames, ignore_index=True)
print(f"Combined dataset: {clean.shape[0]:,} rows × {clean.shape[1]} columns")

Combined dataset: 5,436 rows × 9 columns


In [6]:
print("Reshaped example: first source, US Registered Voters")
display(clean.head())

Reshaped example: first source, US Registered Voters


,approve,disapprove,not_sure,unweighted_base,base,date,group_type,group,source_file
0,0.45,0.49,0.06,1315,1086,2017-02-07,overall,US Registered Voters,donald-trump-approval-2017-2021.xlsx
1,0.46,0.49,0.05,1283,1100,2017-02-14,overall,US Registered Voters,donald-trump-approval-2017-2021.xlsx
2,0.48,0.47,0.05,1198,1031,2017-02-22,overall,US Registered Voters,donald-trump-approval-2017-2021.xlsx
3,0.45,0.51,0.04,1251,1051,2017-03-01,overall,US Registered Voters,donald-trump-approval-2017-2021.xlsx
4,0.44,0.51,0.06,1359,1168,2017-03-07,overall,US Registered Voters,donald-trump-approval-2017-2021.xlsx


## 3. Study window selection

Keep January–June of 2017, 2018, 2025 and 2026 for every group. January 2017 is absent from the source and is not imputed.

In [7]:
rows_before = len(clean)
PERIODS = {2017: "T1_Y1", 2018: "T1_Y2", 2025: "T2_Y1", 2026: "T2_Y2"}
clean["year"] = clean.date.dt.year
clean["month"] = clean.date.dt.month
clean = clean.loc[clean.year.isin(PERIODS) & clean.month.between(1, 6)].copy()
clean["period"] = clean.year.map(PERIODS)
print(f"Before filtering: {rows_before:,} rows")
print(f"After filtering:  {len(clean):,} rows")

Before filtering: 5,436 rows
After filtering:  1,728 rows


In [8]:
coverage = clean.groupby("period").agg(
    rows=("date", "size"), dates=("date", "nunique"),
    groups=("group", "nunique"), months=("month", "nunique"),
    first_date=("date", "min"), last_date=("date", "max"))
display(coverage)

,rows,dates,groups,months,first_date,last_date
period,,,,,,
T1_Y1,378,21,18,5,2017-02-07,2017-06-27
T1_Y2,468,26,18,6,2018-01-02,2018-06-26
T2_Y1,414,23,18,6,2025-01-28,2025-06-30
T2_Y2,468,26,18,6,2026-01-05,2026-06-29


## 4. Data quality checks

In [9]:
SHARES = ["approve", "disapprove", "not_sure"]
BASES = ["unweighted_base", "base"]
KEY = ["date", "group_type", "group"]
clean[SHARES + BASES] = clean[SHARES + BASES].apply(pd.to_numeric)
missing = clean.isna().sum()
duplicate_count = clean.duplicated(KEY).sum()

display(missing.rename("missing_values").to_frame())
print(f"Duplicate date / group_type / group keys: {duplicate_count}")

,missing_values
0,
approve,0
disapprove,0
not_sure,0
unweighted_base,0
base,0
date,0
group_type,0
group,0
source_file,0


Duplicate date / group_type / group keys: 0


In [10]:
display(clean[SHARES + BASES].agg(["min", "max"]).T)
share_totals = clean[SHARES].sum(axis=1)
display((100 * share_totals).round(6).value_counts().sort_index()
        .rename_axis("percentage_total").rename("rows").to_frame())

,min,max
0,,
approve,0.02,0.95
disapprove,0.03,0.97
not_sure,0.00,0.25
unweighted_base,57.00,1693.00
base,44.00,1359.00


,rows
percentage_total,
99.0,214
100.0,1294
101.0,220


In [11]:
assert missing[KEY + ["period", "source_file"] + SHARES + BASES].eq(0).all()
assert ((clean[SHARES] >= 0) & (clean[SHARES] <= 1)).all().all()
assert np.allclose(share_totals, 1, atol=0.010001, rtol=0)
assert (clean[BASES] > 0).all().all()
assert duplicate_count == 0

clean[SHARES] = (100 * clean[SHARES]).round(6)
print("Checks passed. Approval shares are now percentages (0–100).")

Checks passed. Approval shares are now percentages (0–100).


## 5. Final dataset

The cleaned file contains 96 source dates across 18 groups in the four study windows. Select `group_type == "overall"` for the total registered-voter series; subgroup estimates should not be added together.

In [12]:
COLUMNS = ["date", "year", "month", "period", "group_type", "group",
           "approve", "disapprove", "not_sure", "unweighted_base", "base", "source_file"]
clean = clean[COLUMNS].sort_values(KEY).reset_index(drop=True)
print(f"Final shape: {clean.shape[0]:,} rows × {clean.shape[1]} columns")

Final shape: 1,728 rows × 12 columns


In [13]:
display(clean.head())

,date,year,month,period,group_type,group,approve,disapprove,not_sure,unweighted_base,base,source_file
0,2017-02-07,2017,2,T1_Y1,age,30-44,45.0,50.0,6.0,306,236,donald-trump-approval-2017-2021.xlsx
1,2017-02-07,2017,2,T1_Y1,age,45-64,50.0,43.0,7.0,524,391,donald-trump-approval-2017-2021.xlsx
2,2017-02-07,2017,2,T1_Y1,age,65+,51.0,46.0,3.0,233,261,donald-trump-approval-2017-2021.xlsx
3,2017-02-07,2017,2,T1_Y1,age,Under 30,26.0,65.0,9.0,252,197,donald-trump-approval-2017-2021.xlsx
4,2017-02-07,2017,2,T1_Y1,education,College grad,48.0,49.0,4.0,275,230,donald-trump-approval-2017-2021.xlsx


In [14]:
clean.to_csv(OUT, index=False, encoding="utf-8", date_format="%Y-%m-%d")
print(f"Saved: {OUT.relative_to(ROOT)}")

Saved: data\cleaned\yougov_approval_clean.csv
